# Grad-CAM analysis

This notebook examines where the trained classifier obtains evidence for its predictions. The emphasis is on comparing correct decisions, confident errors and recurrently confused species rather than collecting heatmaps without interpretation.

## Setup

The paths mirror the training notebook. The same code can later be pointed at the pretrained checkpoint by changing `EXPERIMENT` and the checkpoint filename.

In [ ]:
from pathlib import Path
import sys

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/cv-project")
    DATA_ROOT = Path("/content/inat500")
    DRIVE_ROOT = Path("/content/drive/MyDrive/COMP9517")
    CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints"
else:
    REPO_ROOT = Path("/Volumes/Fatboi/cv project")
    DATA_ROOT = REPO_ROOT
    CHECKPOINT_ROOT = REPO_ROOT / "checkpoints"

sys.path.insert(0, str(REPO_ROOT / "src"))
EXPERIMENT = "scratch"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "resnet18_scratch_best.pt"
RESULT_ROOT = (
    DRIVE_ROOT / "results" / EXPERIMENT
    if ON_COLAB
    else REPO_ROOT / "results" / EXPERIMENT
)
GRADCAM_ROOT = (
    DRIVE_ROOT / "results/gradcam" / EXPERIMENT
    if ON_COLAB
    else REPO_ROOT / "results/gradcam" / EXPERIMENT
)
GRADCAM_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from inat_project.config import load_config
from inat_project.data import create_dataloaders
from inat_project.evaluation import collect_predictions
from inat_project.gradcam import GradCAM, denormalise_image, overlay_cam
from inat_project.models import create_resnet18
from inat_project.training import get_device, load_checkpoint, seed_everything

In [ ]:
config = load_config(REPO_ROOT / "configs/resnet18_scratch.yaml")
seed_everything(config["seed"])
device = get_device()

with (REPO_ROOT / "metadata/class_to_idx.json").open(encoding="utf-8") as handle:
    class_to_idx = json.load(handle)
idx_to_class = {index: name for name, index in class_to_idx.items()}

model = create_resnet18(
    num_classes=config["data"]["num_classes"],
    pretrained=False,
).to(device)
saved = load_checkpoint(CHECKPOINT_PATH, model, device=device)
if saved["class_to_idx"] != class_to_idx:
    raise ValueError("Checkpoint and dataset label mappings differ.")
model.eval()
print(f"Loaded epoch {saved['epoch'] + 1} on {device}")

In [ ]:
data_config = config["data"]
loaders = create_dataloaders(
    DATA_ROOT,
    REPO_ROOT / "metadata",
    batch_size=data_config["batch_size"],
    image_size=data_config["image_size"],
    num_workers=0 if device.type == "mps" else data_config["num_workers"],
    seed=config["seed"],
    pin_memory=device.type == "cuda",
)
test_dataset = loaders["test"].dataset
path_to_index = {path: index for index, path in enumerate(test_dataset.frame["path"])}
len(test_dataset)

In [ ]:
prediction_path = RESULT_ROOT / "test_predictions.csv"
if prediction_path.exists():
    predictions = pd.read_csv(prediction_path)
else:
    predictions, _ = collect_predictions(model, loaders["test"], device)
    predictions["true_class"] = predictions["true_idx"].map(idx_to_class)
    predictions["predicted_class"] = predictions["predicted_idx"].map(idx_to_class)

print(f"Correct:   {int(predictions['correct'].sum()):,}")
print(f"Incorrect: {int((~predictions['correct'].astype(bool)).sum()):,}")

## Correct and incorrect cases

The target for each heatmap is the model's predicted class. Correct cases are sampled rather than restricted to the easiest images. For errors, high-confidence mistakes are useful because they show where the model found convincing but misleading evidence.

In [ ]:
correct_mask = predictions["correct"].astype(bool)
correct_cases = predictions[correct_mask].sample(
    n=min(6, int(correct_mask.sum())), random_state=config["seed"]
)
incorrect_cases = (
    predictions[~correct_mask]
    .sort_values("confidence", ascending=False)
    .head(6)
)
selected_cases = pd.concat(
    [correct_cases.assign(case_type="correct"),
     incorrect_cases.assign(case_type="incorrect")],
    ignore_index=True,
)
selected_cases[["case_type", "path", "true_idx", "predicted_idx", "confidence"]]

In [ ]:
def short_species(class_name):
    return " ".join(class_name.split("_")[-2:])

figure, axes = plt.subplots(len(selected_cases), 2, figsize=(8, 3.2 * len(selected_cases)))
case_records = []

with GradCAM(model, model.layer4[-1]) as gradcam:
    for row_number, (_, row) in enumerate(selected_cases.iterrows()):
        dataset_index = path_to_index[row["path"]]
        image_tensor, true_index, relative_path = test_dataset[dataset_index]
        batch = image_tensor.unsqueeze(0).to(device)
        target = torch.tensor([int(row["predicted_idx"])], device=device)
        cam, _ = gradcam(batch, target_indices=target)

        original = denormalise_image(image_tensor)
        heatmap = cam[0].cpu().numpy()
        overlay = overlay_cam(original, heatmap)
        true_name = short_species(idx_to_class[int(true_index)])
        predicted_name = short_species(idx_to_class[int(row["predicted_idx"])])

        axes[row_number, 0].imshow(original)
        axes[row_number, 0].set_title(f"True: {true_name}", fontsize=9)
        axes[row_number, 1].imshow(overlay)
        axes[row_number, 1].set_title(
            f"Predicted: {predicted_name} ({row['confidence']:.2f})", fontsize=9
        )
        axes[row_number, 0].axis("off")
        axes[row_number, 1].axis("off")
        case_records.append(
            {
                "case_type": row["case_type"],
                "path": relative_path,
                "true_idx": int(true_index),
                "predicted_idx": int(row["predicted_idx"]),
                "confidence": float(row["confidence"]),
            }
        )

figure.tight_layout()
figure.savefig(GRADCAM_ROOT / "correct_and_incorrect.png", dpi=180, bbox_inches="tight")
pd.DataFrame(case_records).to_csv(GRADCAM_ROOT / "cases.csv", index=False)
plt.show()

## Recurrent confusion pair

The following cell takes the most frequent directed confusion from the test set and produces maps for those mistakes. A directed pair is used because the reverse error rate can be different.

In [ ]:
pairs = pd.read_csv(RESULT_ROOT / "confused_pairs.csv")
pair = pairs.iloc[0]
pair_cases = predictions[
    (predictions["true_idx"] == pair["true_idx"])
    & (predictions["predicted_idx"] == pair["predicted_idx"])
].head(6)
print(
    f"{short_species(pair['true_class'])} → "
    f"{short_species(pair['predicted_class'])}: {int(pair['count'])} test images"
)
pair_cases[["path", "confidence"]]

In [ ]:
figure, axes = plt.subplots(len(pair_cases), 2, figsize=(8, 3.2 * len(pair_cases)))
if len(pair_cases) == 1:
    axes = np.array([axes])

with GradCAM(model, model.layer4[-1]) as gradcam:
    for row_number, (_, row) in enumerate(pair_cases.iterrows()):
        image_tensor, true_index, _ = test_dataset[path_to_index[row["path"]]]
        target = torch.tensor([int(row["predicted_idx"])], device=device)
        cam, _ = gradcam(image_tensor.unsqueeze(0).to(device), target)
        original = denormalise_image(image_tensor)
        overlay = overlay_cam(original, cam[0].cpu().numpy())
        axes[row_number, 0].imshow(original)
        axes[row_number, 1].imshow(overlay)
        axes[row_number, 0].axis("off")
        axes[row_number, 1].axis("off")

figure.suptitle(
    f"{short_species(pair['true_class'])} predicted as {short_species(pair['predicted_class'])}",
    y=1.01,
)
figure.tight_layout()
figure.savefig(GRADCAM_ROOT / "most_confused_pair.png", dpi=180, bbox_inches="tight")
plt.show()

## Interpretation notes

For the report, record observations from the actual maps rather than assuming that bright areas are meaningful. Useful questions are:

- Does the activation overlap the organism, a discriminative part, or mostly the background?
- Are incorrect predictions driven by a visually similar organism, contextual cues, cropping, or poor image quality?
- Does the same region remain important across several images of a confused pair?
- Are the maps diffuse for the scratch model but more localised for the pretrained model?

Apply this notebook to the pretrained checkpoint as well before making the final comparison.